# Valoración de opciones europeas con Black-Scholes-Merton

Este cuaderno calcula el valor de opciones europeas **call** y **put** mediante el modelo Black-Scholes-Merton y muestra cómo cambia su precio al modificar el strike, el vencimiento, el tipo de interés y la volatilidad.

El modelo supone un activo sin dividendos, tipo de interés constante y volatilidad constante.

## 1. Importación de librerías

Se utiliza `scipy.stats.norm` para la distribución normal acumulada. Esto evita implementar integración numérica manual para cada evaluación y permite trabajar también con arrays de NumPy.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=6, suppress=True)


## 2. Funciones auxiliares y validación de parámetros

In [ ]:
def _validate_inputs(S, K, T, r, sigma):
    """Valida los parámetros del modelo y acepta escalares o arrays."""
    values = {
        "S": np.asarray(S, dtype=float),
        "K": np.asarray(K, dtype=float),
        "T": np.asarray(T, dtype=float),
        "r": np.asarray(r, dtype=float),
        "sigma": np.asarray(sigma, dtype=float),
    }

    for name, value in values.items():
        if not np.all(np.isfinite(value)):
            raise ValueError(f"{name} debe contener valores finitos.")

    if np.any(values["S"] <= 0):
        raise ValueError("El precio del activo S debe ser mayor que cero.")
    if np.any(values["K"] <= 0):
        raise ValueError("El strike K debe ser mayor que cero.")
    if np.any(values["T"] <= 0):
        raise ValueError("El vencimiento T debe ser mayor que cero.")
    if np.any(values["sigma"] <= 0):
        raise ValueError("La volatilidad sigma debe ser mayor que cero.")


def _to_scalar_if_needed(value):
    """Devuelve float para resultados escalares y ndarray para resultados vectoriales."""
    array = np.asarray(value)
    return array.item() if array.ndim == 0 else array

## 3. Modelo Black-Scholes-Merton

Las fórmulas utilizadas son:

$$d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}$$

$$d_2 = d_1 - \sigma\sqrt{T}$$

$$C = S N(d_1) - K e^{-rT} N(d_2)$$

$$P = K e^{-rT} N(-d_2) - S N(-d_1)$$

In [ ]:
def d1_d2(S, K, T, r, sigma):
    """Calcula d1 y d2 del modelo Black-Scholes-Merton."""
    _validate_inputs(S, K, T, r, sigma)
    S, K, T, r, sigma = np.broadcast_arrays(
        np.asarray(S, dtype=float),
        np.asarray(K, dtype=float),
        np.asarray(T, dtype=float),
        np.asarray(r, dtype=float),
        np.asarray(sigma, dtype=float),
    )
    volatility_time = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / volatility_time
    d2 = d1 - volatility_time
    return _to_scalar_if_needed(d1), _to_scalar_if_needed(d2)


def european_call_value(S, K, T, r, sigma):
    """Valor presente de una opción call europea sin dividendos."""
    d1, d2 = d1_d2(S, K, T, r, sigma)
    value = np.asarray(S, dtype=float) * norm.cdf(d1) - (
        np.asarray(K, dtype=float) * np.exp(-np.asarray(r, dtype=float) * np.asarray(T, dtype=float)) * norm.cdf(d2)
    )
    return _to_scalar_if_needed(value)


def european_put_value(S, K, T, r, sigma):
    """Valor presente de una opción put europea sin dividendos."""
    d1, d2 = d1_d2(S, K, T, r, sigma)
    value = (
        np.asarray(K, dtype=float) * np.exp(-np.asarray(r, dtype=float) * np.asarray(T, dtype=float)) * norm.cdf(-d2)
        - np.asarray(S, dtype=float) * norm.cdf(-d1)
    )
    return _to_scalar_if_needed(value)


def option_value(option_type, S, K, T, r, sigma):
    """Valora una opción europea de tipo 'call' o 'put'."""
    if option_type.lower() == "call":
        return european_call_value(S, K, T, r, sigma)
    if option_type.lower() == "put":
        return european_put_value(S, K, T, r, sigma)
    raise ValueError("option_type debe ser 'call' o 'put'.")

## 4. Ejemplo de valoración

Parámetros del ejemplo: activo `S=100`, strike `K=100`, vencimiento `T=1` año, tipo de interés `r=5%` y volatilidad `sigma=20%`.

In [ ]:
S = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20

call_value = european_call_value(S, K, T, r, sigma)
put_value = european_put_value(S, K, T, r, sigma)

results = {
    "Opción": ["Call europea", "Put europea"],
    "Valor": [call_value, put_value],
}
results


## 5. Verificación de la paridad put-call

Para opciones europeas sin dividendos se cumple:

$$C - P = S - K e^{-rT}$$

In [ ]:
put_call_difference = call_value - put_value
parity_value = S - K * math.exp(-r * T)

print(f"C - P:                 {put_call_difference:.10f}")
print(f"S - K exp(-rT):        {parity_value:.10f}")
print(f"Paridad verificada:    {math.isclose(put_call_difference, parity_value, rel_tol=1e-10, abs_tol=1e-10)}")

## 6. Visualización de sensibilidades

La siguiente función genera cuatro gráficos para estudiar el efecto de cada variable manteniendo las demás constantes.

In [ ]:
def plot_sensitivities(S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.20, option_type="call"):
    """Representa el valor de una opción frente a sus principales parámetros."""
    option_type = option_type.lower()
    if option_type not in {"call", "put"}:
        raise ValueError("option_type debe ser 'call' o 'put'.")
    pricing_function = european_call_value if option_type == "call" else european_put_value

    strikes = np.linspace(max(1e-6, 0.5 * S), 1.5 * S, 150)
    maturities = np.linspace(0.01, max(2.0, 2.0 * T), 150)
    rates = np.linspace(-0.02, 0.15, 150)
    volatilities = np.linspace(0.01, 0.80, 150)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"Sensibilidades de una opción europea {option_type.lower()}", fontsize=16)

    axes[0, 0].plot(strikes, pricing_function(S, strikes, T, r, sigma))
    axes[0, 0].set(title="Precio frente al strike", xlabel="Strike K", ylabel="Valor")

    axes[0, 1].plot(maturities, pricing_function(S, K, maturities, r, sigma))
    axes[0, 1].set(title="Precio frente al vencimiento", xlabel="Vencimiento T (años)", ylabel="Valor")

    axes[1, 0].plot(rates, pricing_function(S, K, T, rates, sigma))
    axes[1, 0].set(title="Precio frente al tipo de interés", xlabel="Tipo de interés r", ylabel="Valor")

    axes[1, 1].plot(volatilities, pricing_function(S, K, T, r, volatilities))
    axes[1, 1].set(title="Precio frente a la volatilidad", xlabel="Volatilidad sigma", ylabel="Valor")

    for axis in axes.flat:
        axis.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, axes

In [ ]:
plot_sensitivities(S, K, T, r, sigma, option_type="call")
plot_sensitivities(S, K, T, r, sigma, option_type="put")
plt.show()

## 7. Tabla de precios para distintos strikes

La valoración vectorizada permite construir una tabla de escenarios sin repetir manualmente el cálculo.

In [ ]:
scenario_strikes = np.arange(80.0, 121.0, 5.0)
scenario_table = np.column_stack((
    scenario_strikes,
    european_call_value(S, scenario_strikes, T, r, sigma),
    european_put_value(S, scenario_strikes, T, r, sigma),
))
print("Strike       Call        Put")
for strike, call, put in scenario_table:
    print(f"{strike:6.2f}  {call:10.6f}  {put:10.6f}")
scenario_table
